# `EukaryoticToeholdGate` — usage example (trailing-Kozak layout)

A minimal, real, end-to-end run of the single-input eukaryotic toehold switch —
`EukaryoticToeholdGate` in `engine.gates.toehold`. This class is fully implemented
and tested (`tests/engine/gates/test_toehold.py`, 39 passing tests as of this
writing). This is a sibling to [`toehold.ipynb`](toehold.ipynb) (which drives the
gate through a stub `FoldEngine` for fast, dependency-free iteration and covers both
hosts generically) — here we build `EukaryoticToeholdGate` specifically and use the
**real** `FoldEngine` (ViennaRNA) throughout, so every number below is a genuine
fold, not a placeholder.

`ToeholdGate` builds two structurally different eukaryotic layouts (commits
`d754812`, `ee2f5f6`):

* **`"loop"`** — Kozak embedded in the hairpin loop, ported unmodified from the
  prokaryotic mechanism (steric occlusion of the start codon). Unvalidated for a
  eukaryotic toehold specifically — see `KOZAK_LAYOUTS`'s docstring. It also turns
  out Kozak and AUG are *not* adjacent in this layout (a 6 nt stem-closing segment
  sits between them), which breaks the Kozak consensus's own adjacency requirement.
* **`"trailing"`** — Kozak and the start codon sit *after* the closed hairpin
  instead, modelling scanning-ribosome blockage (docs/modalities.md) rather than
  direct start-codon occlusion. This is the layout the team's own eukaryotic
  scripts actually build (`plasmid_prefix + trg_bind_region + loop + stem_down +
  kozak` — Kozak last), and Kozak is genuinely adjacent to AUG here. It also carries
  no trailing `LINKER_SEQUENCE` — cap-dependent scanning initiates the instant the
  40S subunit meets Kozak+AUG, so nothing after the start codon matters to finding
  it, and the payload attaches directly.

This notebook builds the gate restricted to **`"trailing"`** only
(`kozak_layouts=("trailing",)`), since that's the layout that matches our own
established construction and is the newer, less-exercised code path.

No Django, no worker, no pipeline — just the gate class, constructed and called
directly, the way `pipeline.py` would use it internally.

## Setup

In [ ]:
# Put <repo>/src on the path. Search upward from cwd for pyproject.toml so this works
# wherever Jupyter is launched from.
import sys
from pathlib import Path

for _base in (Path.cwd(), *Path.cwd().parents):
    if (_base / "pyproject.toml").exists():
        _src = _base / "src"
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break

from engine.domain import Host, Regulation, SelectedGene, TriggerSet, Constraints
from engine.gates.toehold import EukaryoticToeholdGate
from engine.gates.tools.folding import FoldEngine
from engine.gates.tools.translation import TranslationScorer
from engine.gates.tools.codons import CodonOptimizer
from engine.stages.folding import FoldProfiler
from engine.stages.motifs import MotifScreener
from engine.stages.off_target import OffTargetScanner
from engine.stages.triggers import TriggerScorer
from engine import sequences as sq

## 1. Build the tools, once

Per `CLAUDE.md` §5: tools are constructed once and handed to the gate, never built
inside a stage or family. `FoldEngine`'s cache is only useful if every caller shares
one instance — a second `FoldEngine()` means a cold cache and, worse, a second
chance to fold at a different temperature.

In [ ]:
host = Host.HUMAN  # HUMAN | YEAST both take the eukaryotic (Kozak) track

folder = FoldEngine(temperature=37.0)
translation = TranslationScorer(host)
codons = CodonOptimizer(host)

# kozak_layouts restricts generate_designs to the "trailing" layout only — the default
# (omit this argument) sweeps both "loop" and "trailing" and lets engine.scoring rank
# across them. This mirrors how `host` is a constructor parameter rather than a
# subclass (docs/engine.md §2.4).
gate = EukaryoticToeholdGate(host, folder, translation, codons, kozak_layouts=("trailing",))
print(gate.required_tools())
print("kozak_layouts:", gate.kozak_layouts)

## 2. Pick a trigger — via the real `TriggerScorer` (stage 2)

`TriggerScorer.score` is fully implemented (`tests/engine/test_triggers.py`, 47
passing tests): it slides every window of every length in
`constraints.trigger_lengths` across the transcript, screens out forbidden motifs,
folds each survivor for `openness`/`accessibility`/`mfe` via the same `FoldEngine`
the gate uses, checks off-targets, ranks by `accessibility * segment_specificity`,
and yields the top candidates per gene — so we use it for real here instead of
hand-picking one window.

`OffTargetScanner` itself is still a stub (`find_similar`/`scan_trigger` raise
`NotImplementedError`) — **except** when handed an empty transcriptome, which is a
deliberate early-return for exactly this case (a `direct` submission, or a demo like
this one, with no reference index to scan against): `scan_trigger` returns a clean
`OffTargetReport(hits=(), penalty=0.0)` rather than raising. That is a real,
documented behaviour of the class, not a workaround.

In [ ]:
# The full-length human AREG mRNA, as cDNA/DNA notation (T, not U) — same alphabet trap
# CLAUDE.md warns about, so normalise with to_rna() before anything else touches it.
transcript_dna = (
    "AGACGTTCGCACACCTGGGTGCCAGCGCCCCAGAGGTCCCGGGACAGCCCGAGGCGCCGCGCCCGCCGCCCCGAGCTCCCC"
    "AAGCCTTCGAGAGCGGCGCACACTCCCGGTCTCCACTCGCTCTTCCAACACCCGCTCGTTTTGGCGGCAGCTCGTGTCCCA"
    "GAGACCGAGTTGCCCCAGAGACCGAGACGCCGCCGCTGCGAAGGACCAATGAGAGCCCCGCTGCTACCGCCGGCGCCGGTG"
    "GTGCTGTCGCTCTTGATACTCGGCTCAGGCCATTATGCTGCTGGATTGGACCTCAATGACACCTACTCTGGGAAGCGTGAA"
    "CCATTTTCTGGGGACCACAGTGCTGATGGATTTGAGGTTACCTCAAGAAGTGAGATGTCTTCAGGGAGTGAGATTTCCCCT"
    "GTGAGTGAAATGCCTTCTAGTAGTGAACCGTCCTCGGGAGCCGACTATGACTACTCAGAAGAGTATGATAACGAACCACAA"
    "ATACCTGGCTATATTGTCGATGATTCAGTCAGAGTTGAACAGGTAGTTAAGCCCCCCCAAAACAAGACGGAAAGTGAAAAT"
    "ACTTCAGATAAACCCAAAAGAAAGAAAAAGGGAGGCAAAAATGGAAAAAATAGAAGAAACAGAAAGAAGAAAAATCCATGT"
    "AATGCAGAATTTCAAAATTTCTGCATTCACGGAGAATGCAAATATATAGAGCACCTGGAAGCAGTAACATGCAAATGTCA"
    "GCAAGAATATTTCGGTGAACGGTGTGGGGAAAAGTCCATGAAAACTCACAGCATGATTGACAGTAGTTTATCAAAAATTG"
    "CATTAGCAGCCATAGCTGCCTTTATGTCTGCTGTGATCCTCACAGCTGTTGCTGTTATTACAGTCCAGCTTAGAAGACAA"
    "TACGTCAGGAAATATGAAGGAGAAGCTGAGGAACGAAAGAAACTTCGACAAGAGAATGGAAATGTACATGCTATAGCATA"
    "ACTGAAGATAAAATTACAGGATATCACATTGGAGTCACTGCCAAGTCATAGCCATAAATGATGAGTCGGTCCTCTTTCCA"
    "GTGGATCATAAGACAATGGACCCTTTTTGTTATGATGGTTTTAAACTTTCAATTGTCACTTTTTATGCTATTTCTGTATA"
    "TAAAGGTGCACGAAGGTAAAAAGTATTTTTTCAAGTTGTAAATAATTTATTTAATATTTAATGGAAGTGTATTTATTTTA"
    "CAGCTCATTAAACTTTTTTAACCAAA"
)
transcript = sq.to_rna(transcript_dna)
assert sq.is_valid_rna(transcript)
print(f"transcript length: {len(transcript)} nt")

In [ ]:
# Stage-2 tools, built once (same injection rule as the gate's own tools).
profiler = FoldProfiler()
screener = MotifScreener()
off_target = OffTargetScanner(transcriptome={})  # empty: no reference index for this demo
scorer = TriggerScorer(profiler, off_target, screener, folder)  # shares the gate's FoldEngine

# In a real run this comes from GeneSelector (stage 1); stand in with a minimal
# SelectedGene since this demo starts from a single known transcript.
gene = SelectedGene(
    gene_id="AREG",
    symbol="AREG",
    regulation=Regulation.UP,
    log2_fold_change=2.0,
    score=1.0,
)
constraints = Constraints(trigger_lengths=(30, 36), max_switch_length=200)

candidates = list(scorer.score([gene], {"AREG": transcript}, constraints))
print(f"{len(candidates)} candidate(s), best-scoring first\n")
for c in candidates[:5]:
    print(
        f"  {c.trigger_id:22} start={c.start_index:4} len={c.length:2}  "
        f"score={c.score:.3f}  accessibility={c.accessibility:.3f}  gc={c.gc_content:.1f}"
    )

trigger = candidates[0]
print("\nselected:", trigger)

## 3. Wrap the trigger in a `TriggerSet`

`TriggerSet` is the circuit's inputs (one activator here — a single-input switch).
`Constraints` were already built above, since `TriggerScorer` needed them too — a run
builds `Constraints` once from `params["constraints"]` and threads the same object
through every stage.

In [ ]:
triggers = TriggerSet(activators=(trigger,))

print("arity:", triggers.arity, "| logic:", triggers.logic_type)

## 4. `is_compatible()` — cheap check before generating anything

Arity, host, trigger length window — nothing here folds.

In [ ]:
compatibility = gate.is_compatible(triggers, constraints)
print(compatibility)
assert compatibility.ok, compatibility.reason

## 5. `generate_designs()` — candidate switches

A generator. With `kozak_layouts=("trailing",)`, one `GateDesign` per combination of
`toehold_lengths` x `TRAILING_LOOP_LENGTHS` x `KOZAK_LINKER_LENGTHS` that this trigger
supports — the loop's length and the optional spacer before Kozak are both swept
design axes for this layout, not fixed constants (see `KOZAK_LAYOUTS`'s docstring for
why). Materialise with `list(...)` here; a real run would consume this lazily.

In [ ]:
designs = list(gate.generate_designs(triggers, constraints))
print(f"{len(designs)} design(s)\n")
for d in designs:
    print(
        f"  {d.design_id:52} {d.length:3} nt  "
        f"loop_len={d.architecture['loop_len']:2}  "
        f"kozak_linker_len={d.architecture['kozak_linker_len']}"
    )

## 6. `evaluate_design()` — raw metrics, and comparing across the swept designs

**Raw** values only — no normalising, weighting or ranking here, that is
`engine.scoring`'s job. Keys are exactly the metric names `DEFAULT_V1` declares.

For the `"trailing"` layout specifically, `predicted_leakage`/`dynamic_range` are read
from the **toehold+stem region**, not the AUG — the AUG sits outside the hairpin here
and stays roughly accessible whether or not the trigger is bound, so AUG-region
accessibility would not discriminate ON from OFF for this layout (see
`evaluate_design`'s docstring). This is a new, unreviewed proxy — not a port of
anything previously validated.

Since `generate_designs` yielded 4 designs (one per `loop_len` x `kozak_linker_len`
combination) and none of them is picked as "the" design here, evaluate every one and
compare — this is what `engine.scoring` would do across the whole candidate pool in a
real run.

In [ ]:
all_metrics = [(d, gate.evaluate_design(d)) for d in designs]

print(
    f"{'loop_len':>8}  {'linker_len':>10}  {'leakage':>8}  {'dyn_range':>9}  "
    f"{'folding_energy':>14}  {'gc':>6}"
)
for d, m in all_metrics:
    print(
        f"{d.architecture['loop_len']:>8}  {d.architecture['kozak_linker_len']:>10}  "
        f"{m['predicted_leakage']:>8.3f}  {m['dynamic_range']:>9.3f}  "
        f"{m['gate_folding_energy']:>14.1f}  {m['gc_content']:>6.1f}"
    )

# Not this gate's job to rank in a real run (engine.scoring owns that) — but picking one
# here to carry through the rest of this notebook's cells.
design, metrics = max(all_metrics, key=lambda pair: pair[1]["dynamic_range"])
print(f"\nbest by dynamic_range: {design.design_id}")
for name, value in metrics.items():
    print(f"  {name:22} {value}")

`predicted_leakage` comes in well under the 0.85 hard-filter threshold across all four
combinations — the toehold+stem region is genuinely well-paired in the OFF state. But
`dynamic_range` (higher is better, per `DEFAULT_V1`) comes out **below 1.0** for every
combination here — the region gets *less* accessible, not more, when the trigger binds.
That is a real result, not a bug: for this particular trigger and `toehold_length=12`
(the shortest swept length, and this trigger's shortest length that clears the
footprint), the `"trailing"` hairpin does not open the way it should on trigger
binding. It is exactly the kind of design `engine.scoring`'s ranking would push to the
bottom of the pool, and exactly why nothing here hand-picks "the" answer — a poor
result at one sweep point is informative, not something to hide by re-running until a
better number appears. Try a longer `toehold_length` sweep, or a different transcript
region, to see whether the mechanism performs better elsewhere.

## 7. `emit_sequence()` — the synthesis-ready sequence

In [ ]:
print(gate.emit_sequence(design))

The Kozak element and start codon aren't visually obvious in that raw string — for this
`"trailing"` layout they sit right after the closed hairpin, not inside it (contrast
with `"loop"`, where they'd be buried in the middle). Note also that the sequence ends
**exactly** at the AUG — unlike `"loop"`, this layout carries no trailing
`LINKER_SEQUENCE` (commit `ee2f5f6`): cap-dependent scanning initiates the instant the
40S subunit meets Kozak+AUG, so nothing after the start codon plays any role in
finding it, and the payload attaches directly here at plasmid assembly. Locate Kozak
and the start codon explicitly using `design.architecture` (which records `aug_index`
exactly) and the gate's own `KOZAK_EUKARYOTIC` constant:

In [ ]:
seq = design.sequence
aug_index = design.architecture["aug_index"]
kozak_index = seq.find(gate.KOZAK_EUKARYOTIC)
assert kozak_index != -1, "Kozak element not found — architecture assumptions above are stale"

marks = [" "] * len(seq)
for i in range(kozak_index, kozak_index + len(gate.KOZAK_EUKARYOTIC)):
    marks[i] = "K"
for i in range(aug_index, aug_index + 3):
    marks[i] = "A"

print(seq)
print("".join(marks), " K = Kozak (GCCACC)   A = start codon (AUG)")

## Where this fits in a real run

This notebook stops at one layout, one gate, a handful of designs. In the actual
pipeline:

- `generate_designs` is called for **every** trigger set that passed `is_compatible`,
  and by default sweeps **both** `"loop"` and `"trailing"` layouts (this notebook
  restricted to `"trailing"` only via `kozak_layouts` — see cell 4) — yielding
  potentially thousands of designs across both mechanisms.
- Every design's raw metrics go through `engine.scoring` — `build_metrics`,
  `weighted_score`, `failed_filter`, `rank_candidates` — which is what actually
  decides which designs survive and how they rank, comparably with every other gate
  family's designs **and across layouts**. Neither this notebook nor the gate itself
  picks a winner between `"loop"` and `"trailing"`.
- `CandidateStore` records provenance and writes the stage snapshot; nothing here
  hand-rolls a results CSV.
- The **payload** (your actual effector gene) is not part of `GateDesign.sequence` at
  all, for either layout — it attaches later, at plasmid assembly
  (`PlasmidBuilder.build(circuit, DesiredOutcome.CUSTOM, custom_payload=...)`). For
  `"trailing"`, `design.sequence[aug_index:]` is exactly `"AUG"`, so the payload fuses
  immediately after the switch's own start codon with no spacer in between.

The two-input AND version (`EukaryoticToeholdAndGate`) is **not** implemented yet —
its `generate_designs` still raises `NotImplementedError("Step 5")`. This notebook
only covers the single-input case.

Open questions this layout surfaced, not resolved here (see commits `d754812`,
`ee2f5f6`): whether `"loop"` should still ship as the eukaryotic default now that
`"trailing"` exists (and now that `"loop"`'s Kozak-AUG adjacency is known to be
broken); whether `TRAILING_LOOP_LENGTHS`/`KOZAK_LINKER_LENGTHS` are the right ranges
to sweep; whether the `"trailing"` leakage proxy (toehold+stem accessibility) is the
right measurement for scanning-ribosome blockage at all; and whether fusing the
payload directly after the switch's own placeholder AUG (rather than dropping that
AUG in favour of the payload's own) is the right call — the fused ORF currently reads
switch-AUG then the payload's required leading AUG as an ordinary internal codon.